In [ ]:
#!/usr/bin/env python3
# /// script
# requires-python = ">=3.13"
# dependencies = [
#   "braid",
#   "icechunk",
#   "obspec-utils",
#   "obstore",
#   "virtualizarr",
# ]
#
# [tool.uv.sources]
# braid = { path = ".." }
# ///
"""
Virtualize GOES-16 ABI-L2-MCMIPF files in parallel using braid, writing to
an IceChunk virtual store via concat + append writes.

Usage:
    uv run python examples/goes_1000.py

Requires:
    - braid initialized: `braid init --bucket <bucket> ...`
    - braid synced: `braid sync`
    - AWS credentials with Lambda + S3 + SQS access
"""

import time
import xarray as xr 
import icechunk as ic
from braid import fan
from virtualizarr.writers.icechunk import virtual_dataset_to_icechunk

GOES_BUCKET = "s3://noaa-goes16"
GOES_REGION = "us-east-1"

ICECHUNK_BUCKET = "carbonplan-scratch"
ICECHUNK_PREFIX = "goes16/goes16.icechunk"
ICECHUNK_REGION = "us-west-2"

BRAID_BATCH = 10      # Each Lambda processes 10 files (saves Lambda overhead)
COMMIT_EVERY = 500    # Checkpoint every 500 files
# Spatial data variables to keep; all others dropped via drop_variables.
_KEEP_VARIABLES = {
    *(f"CMI_C{i:02}" for i in range(1, 17)),
    *(f"DQF_C{i:02}" for i in range(1, 17)),
    "x", "y", "t",
}

keep_variables = [
    'CMI_C01',
 'CMI_C02',
 'CMI_C03',
 'CMI_C04',
 'CMI_C05',
 'CMI_C06',
 'CMI_C07',
 'CMI_C08',
 'CMI_C09',
 'CMI_C10',
 'CMI_C11',
 'CMI_C12',
 'CMI_C13',
 'CMI_C14',
 'CMI_C15',
 'CMI_C16',
 'DQF_C01',
 'DQF_C02',
 'DQF_C03',
 'DQF_C04',
 'DQF_C05',
 'DQF_C06',
 'DQF_C07',
 'DQF_C08',
 'DQF_C09',
 'DQF_C10',
 'DQF_C11',
 'DQF_C12',
 'DQF_C13',
 'DQF_C14',
 'DQF_C15',
 'DQF_C16',
 't',
 'x',
 'y'
]
drop_variables = ['outlier_pixel_count_C13',
 'std_dev_reflectance_factor_C04',
 'max_brightness_temperature_C10',
 'max_reflectance_factor_C06',
 'mean_reflectance_factor_C02',
 'outlier_pixel_count_C03',
 'max_brightness_temperature_C15',
 'min_reflectance_factor_C03',
 'std_dev_reflectance_factor_C03',
 'std_dev_brightness_temperature_C12',
 'std_dev_brightness_temperature_C09',
 'y_image_bounds',
 'std_dev_brightness_temperature_C16',
 'outlier_pixel_count_C16',
 'percent_uncorrectable_GRB_errors',
 'min_brightness_temperature_C15',
 'max_brightness_temperature_C09',
 'max_brightness_temperature_C14',
 'outlier_pixel_count_C01',
 'mean_brightness_temperature_C15',
 'mean_brightness_temperature_C13',
 'outlier_pixel_count_C09',
 'band_id_C05',
 'outlier_pixel_count_C02',
 'std_dev_brightness_temperature_C13',
 'band_wavelength_C02',
 'mean_reflectance_factor_C06',
 'min_brightness_temperature_C07',
 'mean_brightness_temperature_C16',
 'band_wavelength_C03',
 'mean_brightness_temperature_C12',
 'geospatial_lat_lon_extent',
 'min_reflectance_factor_C05',
 'std_dev_brightness_temperature_C14',
 'band_id_C14',
 'max_brightness_temperature_C11',
 'max_reflectance_factor_C02',
 'max_brightness_temperature_C07',
 'band_id_C08',
 'max_brightness_temperature_C08',
 'band_wavelength_C09',
 'band_wavelength_C08',
 'mean_reflectance_factor_C01',
 'mean_brightness_temperature_C14',
 'algorithm_product_version_container',
 'band_wavelength_C11',
 'mean_brightness_temperature_C11',
 'max_brightness_temperature_C13',
 'nominal_satellite_subpoint_lon',
 'mean_brightness_temperature_C07',
 'band_id_C06',
 'std_dev_brightness_temperature_C11',
 'outlier_pixel_count_C15',
 'x_image',
 'band_wavelength_C14',
 'min_brightness_temperature_C09',
 'max_reflectance_factor_C01',
 'std_dev_reflectance_factor_C06',
 'time_bounds',
 'std_dev_reflectance_factor_C02',
 'band_id_C04',
 'outlier_pixel_count_C04',
 'std_dev_reflectance_factor_C01',
 'mean_reflectance_factor_C03',
 'min_brightness_temperature_C16',
 'mean_reflectance_factor_C04',
 'outlier_pixel_count_C07',
 'min_brightness_temperature_C08',
 'min_reflectance_factor_C04',
 'min_brightness_temperature_C14',
 'outlier_pixel_count_C11',
 'min_brightness_temperature_C13',
 'std_dev_brightness_temperature_C10',
 'min_brightness_temperature_C10',
 'percent_uncorrectable_L0_errors',
 'mean_reflectance_factor_C05',
 'outlier_pixel_count_C05',
 'outlier_pixel_count_C12',
 'band_wavelength_C07',
 'band_id_C09',
 'min_brightness_temperature_C11',
 'min_reflectance_factor_C02',
 'band_wavelength_C10',
 'band_wavelength_C16',
 'band_id_C16',
 'band_id_C03',
 'band_id_C01',
 'band_wavelength_C12',
 'max_reflectance_factor_C03',
 'min_reflectance_factor_C01',
 'mean_brightness_temperature_C09',
 'band_wavelength_C05',
 'outlier_pixel_count_C06',
 'std_dev_brightness_temperature_C15',
 'outlier_pixel_count_C10',
 'max_brightness_temperature_C16',
 'dynamic_algorithm_input_data_container',
 'band_id_C10',
 'mean_brightness_temperature_C10',
 'std_dev_brightness_temperature_C08',
 'nominal_satellite_height',
 'outlier_pixel_count_C14',
 'band_id_C02',
 'band_id_C15',
 'x_image_bounds',
 'band_wavelength_C13',
 'band_wavelength_C04',
 'band_id_C12',
 'max_brightness_temperature_C12',
 'min_brightness_temperature_C12',
 'y_image',
 'max_reflectance_factor_C04',
 'nominal_satellite_subpoint_lat',
 'band_wavelength_C06',
 'band_wavelength_C15',
 'band_id_C07',
 'goes_imager_projection',
 'std_dev_brightness_temperature_C07',
 'band_id_C11',
 'min_reflectance_factor_C06',
 'band_wavelength_C01',
 'std_dev_reflectance_factor_C05',
 'max_reflectance_factor_C05',
 'outlier_pixel_count_C08',
 'mean_brightness_temperature_C08',
 'band_id_C13']
 
def _repo_storage():
    return ic.s3_storage(bucket=ICECHUNK_BUCKET, prefix=ICECHUNK_PREFIX, region=ICECHUNK_REGION)


def _repo_config() -> ic.RepositoryConfig:
    config = ic.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        ic.VirtualChunkContainer(f"{GOES_BUCKET}/", store=ic.s3_store(region=GOES_REGION))
    )
    # Split manifests by t-dimension: each commit only rewrites shards for new t-indices,
    # not the full manifest. Keeps commit time O(batch_size) not O(total_store_size).
    split_cfg = ic.ManifestSplittingConfig.from_dict({
        ic.ManifestSplitCondition.AnyArray(): {
            ic.ManifestSplitDimCondition.DimensionName("t"): COMMIT_EVERY
        }
    })
    config.manifest = ic.ManifestConfig(splitting=split_cfg)
    return config


def _open_or_create_repo() -> ic.Repository:
    credentials = ic.containers_credentials({f"{GOES_BUCKET}/": None})
    return ic.Repository.open_or_create(
        _repo_storage(), config=_repo_config(), authorize_virtual_chunk_access=credentials
    )


def _find_resume_index(repo) -> int:
    for commit in repo.ancestry(branch="main"):
        if commit.metadata and "last_hi" in commit.metadata:
            return int(commit.metadata["last_hi"])
    return 0


def list_goes_files(n: int | None = None, cache_path: str = "/tmp/goes16_2024_paths.txt") -> list[str]:
    import asyncio
    from pathlib import Path

    import obstore as obs
    from obstore.store import from_url

    p = Path(cache_path)
    if p.exists():
        paths = p.read_text().splitlines()
        print(f"Loaded {len(paths)} paths from cache ({cache_path})")
        return paths[:n] if n else paths

    async def _list_all() -> list[str]:
        store = from_url(GOES_BUCKET, region=GOES_REGION, skip_signature=True)
        top = await obs.list_with_delimiter_async(store, prefix="ABI-L2-MCMIPF/2024/")
        day_prefixes = top["common_prefixes"]

        async def list_day(prefix: str) -> list[str]:
            return [
                item["path"]
                for batch in obs.list(store, prefix=prefix + "/")
                for item in batch
                if item["path"].endswith(".nc")
            ]

        results = await asyncio.gather(*[list_day(p) for p in day_prefixes])
        return sorted(p for day in results for p in day)

    print("Listing all GOES-16 ABI-L2-MCMIPF 2024 files...")
    paths = asyncio.run(_list_all())
    p.write_text("\n".join(paths))
    print(f"Found {len(paths)} files, cached to {cache_path}")
    return paths[:n] if n else paths


def open_vds(path: str, drop_variables: list[str] | None = None):
    """Open a GOES-16 file via VirtualiZarr HDFParser.

    Returns virtual dataset: CMI/DQF as ManifestArrays (dims y,x), t as numpy scalar,
    x/y as indexed numpy coords.
    """
    from obspec_utils.registry import ObjectStoreRegistry
    from obstore.store import from_url
    from virtualizarr import open_virtual_dataset
    from virtualizarr.parsers import HDFParser

    store = from_url(GOES_BUCKET, region=GOES_REGION, skip_signature=True)
    registry = ObjectStoreRegistry({GOES_BUCKET: store})
    vds = open_virtual_dataset(
        url=f"{GOES_BUCKET}/{path}",
        parser=HDFParser(),
        registry=registry,
        loadable_variables=["t","y","x"],
        drop_variables=drop_variables,
    )
    vds = vds.expand_dims("t")
    vds['t'] = vds['t'].dt.strftime("%Y-%m-%d %H:%M:%S")
    return vds


In [ ]:
all_paths = list_goes_files(n=12)
n = len(all_paths)
repo = _open_or_create_repo()
resume_from = _find_resume_index(repo)

In [ ]:
resume_from

In [ ]:

resume_from = _find_resume_index(repo)
remaining = list(range(resume_from, n))
batches = [remaining[i:i + COMMIT_EVERY] for i in range(0, len(remaining), COMMIT_EVERY)]
print(f"\nFanning {len(remaining)} files in {len(batches)} batches of {COMMIT_EVERY}...")


In [ ]:
import functools 

_open_vds = functools.partial(open_vds, drop_variables=drop_variables)


for batch in batches:
    lo, hi = batch[0], batch[-1] + 1
    print(f"\n── Batch files {lo}-{hi} ──")

    t1 = time.time()
    results=[]
    for vds_list in fan(
        _open_vds,
        [all_paths[i] for i in batch],
        batch_size=BRAID_BATCH,
        timeout_s=900,
        # exclude_packages=["scipy"],
    ):
        results.append(vds_list)
    print(f"Fanned {len(results)} files in {time.time() - t1:.1f}s")
    concat_ds = xr.concat(results, dim="t")

    session = repo.writable_session("main")
    if resume_from == 0:
        virtual_dataset_to_icechunk(concat_ds, session.store)
    else:
        virtual_dataset_to_icechunk(concat_ds, session.store, append_dim="t")

    session.commit(f"goes16 t={lo}:{hi} of {n}", metadata={"last_hi": str(hi)})
    # print(f"Wrote + committed in {time.time() - t2:.1f}s")

In [ ]:
resume_from == 0

In [ ]:
concat_ds = xr.concat(results, dim="t")


In [ ]:
vds_list

In [ ]:
results

In [ ]:
vds_list

In [ ]:
vds

In [ ]:


for batch in batches:
    lo, hi = batch[0], batch[-1] + 1
    print(f"\n── Batch files {lo}-{hi} ──")

    t1 = time.time()
    results: dict[int, object] = {}
    for t_idx, vds in fan(
        _open_vds,
        [all_paths[i] for i in batch],
        batch_size=BRAID_BATCH,
        timeout_s=900,
        # exclude_packages=["scipy"],
    ):
        results[t_idx] = vds
    print(f"Fanned {len(results)} files in {time.time() - t1:.1f}s")

    t2 = time.time()
    vds_list = [results[i].expand_dims("t") for i in range(len(batch))]
    concat_ds = xr.concat(vds_list, dim="t")

    session = repo.writable_session("main")
    virtual_dataset_to_icechunk(concat_ds, session.store, append_dim="t")
    session.commit(f"goes16 t={lo}:{hi} of {n}", metadata={"last_hi": str(hi)})
    print(f"Wrote + committed in {time.time() - t2:.1f}s")


if __name__ == "__main__":
main()

In [ ]:
concat_ds = xr.concat(results, dim="t")


In [ ]:
concat_ds

In [ ]:
vds_list